# Phase 1 — Dataset Understanding

This notebook inspects the original 5,010-row CSV before cleaning. It documents the schema, missingness, repeated product titles, and the limits of the supplied category fields. Inferred product categories are exploratory labels, not ground truth.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load original dataset
df = pd.read_csv("../backend/electronics.csv")

# Basic overview
print("DATASET OVERVIEW")
print("-" * 50)

print("Shape:", df.shape)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 records:")
display(df.head())

print("\nStatistical summary:")
display(df.describe(include="all").T)

print("\nMissing values:")
missing = df.isnull().sum()
display(missing[missing > 0].sort_values(ascending=False))

print("\nDuplicate records:", df.duplicated().sum())

DATASET OVERVIEW
--------------------------------------------------
Shape: (5010, 10)
Rows: 5010
Columns: 10

Column names:
['name', 'main_category', 'sub_category', 'image', 'link', 'no_of_ratings', 'discount_price', 'actual_price', 'review_rating', 'review_text']

Data types:
name              object
main_category     object
sub_category      object
image             object
link              object
no_of_ratings     object
discount_price    object
actual_price      object
review_rating     object
review_text       object
dtype: object

First 5 records:
                                                                                                                            name        main_category     sub_category                                                           image                                                                                                                               link no_of_ratings discount_price actual_price       review_rating                

In [3]:
print("MAIN CATEGORY DISTRIBUTION")
display(df["main_category"].value_counts())

print("\nSUBCATEGORY DISTRIBUTION")
display(df["sub_category"].value_counts())

print("\nMOST COMMON PRODUCT NAMES")
display(df["name"].value_counts().head(20))

print("\nSAMPLE PRODUCT TITLES")
for title in df["name"].sample(
    min(20, len(df)),
    random_state=42
):
    print("-", title)

MAIN CATEGORY DISTRIBUTION
main_category
tv, audio & cameras    5010

SUBCATEGORY DISTRIBUTION
sub_category
All Electronics    5010

MOST COMMON PRODUCT NAMES
name
TAGG Verve Sense Smartwatch with 1.70'' Large Display, Real SPO2, and Real-Time Heart Rate Tracking, 7 Days Battery Backu...     7
Fire-Boltt Visionary 1.78" AMOLED Bluetooth Calling Smartwatch with 368 * 448 Pixel Resolution, Rotating Crown & 60Hz Refr...    6
Tarkan Portable Folding Laptop Desk for Bed, Lapdesk with Handle, Drawer, Cup & Mobile/Tablet Holder for Study, Eating, Wo...    5
Fire-Boltt Eterno 1.99" Largest Display, Bluetooth Calling Smartwatch with 1184mm Display Area, AI Voice Assistant, 120+ S...    5
Fire-Boltt Ninja Call Pro Plus 1.83" Smart Watch with Bluetooth Calling, AI Voice Assistance, 100 Sports Modes IP67 Ratin...     5
Logitech Pebble Wireless Mouse with Bluetooth or 2.4 GHz Receiver, Silent, Slim Computer Mouse with Quiet Clicks, for Lapt...    5
Fire-Boltt Invincible Plus 1.43" AMOLED Display Sm

In [4]:
missing_percent = (
    df.isnull().sum() / len(df)
) * 100

missing_report = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": missing_percent
})

display(
    missing_report.sort_values(
        "Missing Percentage",
        ascending=False
    )
)

plt.figure(figsize=(10, 5))

missing_report["Missing Percentage"].plot(
    kind="bar"
)

plt.title("Missing Values by Column")
plt.ylabel("Missing Percentage (%)")
plt.xlabel("Columns")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

                Missing Count  Missing Percentage
discount_price            270            5.389222
no_of_ratings              34            0.678643
actual_price               31            0.618762
review_text                 3            0.059880
name                        0            0.000000
main_category               0            0.000000
sub_category                0            0.000000
image                       0            0.000000
link                        0            0.000000
review_rating               0            0.000000

[stderr]
01_data_understanding.ipynb:cell3:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [5]:
print("DISCOUNT PRICE EXAMPLES")
display(df["discount_price"].head(15))

print("\nACTUAL PRICE EXAMPLES")
display(df["actual_price"].head(15))

print("\nRATING EXAMPLES")
display(df["review_rating"].head(15))

print("\nREVIEW COUNT EXAMPLES")
display(df["no_of_ratings"].head(15))

DISCOUNT PRICE EXAMPLES
0     ₹10,999
1     ₹18,999
2      ₹1,999
3     ₹15,999
4     ₹18,999
5     ₹10,999
6      ₹1,299
7      ₹1,589
8      ₹1,799
9     ₹15,999
10     ₹9,749
11     ₹5,899
12    ₹20,999
13     ₹1,299
14     ₹1,299

ACTUAL PRICE EXAMPLES
0     ₹18,999
1     ₹19,999
2      ₹2,299
3     ₹24,999
4     ₹19,999
5     ₹18,999
6      ₹3,995
7      ₹1,900
8     ₹19,999
9     ₹24,999
10    ₹13,499
11     ₹8,999
12    ₹21,999
13     ₹4,490
14     ₹4,490

RATING EXAMPLES
0     3.0 out of 5 stars
1     1.0 out of 5 stars
2     5.0 out of 5 stars
3     1.0 out of 5 stars
4     5.0 out of 5 stars
5     1.0 out of 5 stars
6     5.0 out of 5 stars
7     1.0 out of 5 stars
8     4.0 out of 5 stars
9     1.0 out of 5 stars
10    3.0 out of 5 stars
11    1.0 out of 5 stars
12    5.0 out of 5 stars
13    1.0 out of 5 stars
14    5.0 out of 5 stars

REVIEW COUNT EXAMPLES
0         965
1     113,956
2      90,304
3      24,863
4     113,956
5         625
6     172,347
7      61,748
8     

In [6]:
import sys
sys.path.insert(0, "..")
from backend.product_features import classify_product

df["detected_category"] = df["name"].map(classify_product)
display(df["detected_category"].value_counts().to_frame("Count"))
print("These labels are rule-based title estimates, not source-provided ground truth.")

                                Count
detected_category                    
Other / Unclassified             1007
Headphones and Earphones          597
Mobile Accessories                532
Chargers and Cables               482
Smartphones                       408
Computer Accessories              330
Smartwatches                      214
Laptops                           187
Batteries                         186
Tablets                           144
Storage Devices                   130
Non-electronic / Miscellaneous    120
Speakers                           98
Remote Controls                    91
Printers and Printing Supplies     84
Camera Accessories                 60
Power Banks                        55
Televisions                        51
Networking Devices                 47
Monitors and Projectors            42
Cameras                            31
Smartwatch Accessories             30
Electrical Accessories             30
Telephones                         19
Computer Com

In [7]:
repeated_products = (
    df[df.duplicated("name", keep=False)]
    .sort_values("name")
)

print("Repeated product rows:", len(repeated_products))

display(
    repeated_products[
        ["name", "discount_price", "actual_price", "review_rating"]
    ].head(30)
)

Repeated product rows: 685
                                                                                                                               name discount_price actual_price       review_rating
2629  Action Pro™ Helmet Chin Mount Strap Compatible with All Smartphones GoPro Hero 11 10 9 8 7 6 5 4 SJCAM DJI Osmo and All Ac...           ₹335         ₹799  4.0 out of 5 stars
2793  Action Pro™ Helmet Chin Mount Strap Compatible with All Smartphones GoPro Hero 11 10 9 8 7 6 5 4 SJCAM DJI Osmo and All Ac...           ₹335         ₹799  5.0 out of 5 stars
4942  Action Pro™ Helmet Chin Mount Strap Compatible with All Smartphones GoPro Hero 11 10 9 8 7 6 5 4 SJCAM DJI Osmo and All Ac...           ₹335         ₹799  5.0 out of 5 stars
4231  Adofys Flexible Gorillapod Tripod with 360° Rotating Ball Head Tripod for All DSLR Cameras(Max Load 1.5 kgs) & Mobile Phon...           ₹599       ₹2,699  5.0 out of 5 stars
3947  Adofys Flexible Gorillapod Tripod with 360° Rotating Ball Head Trip

In [8]:
repeated_analysis = (
    df.groupby("name")
    .agg(
        occurrences=("name", "size"),
        different_discount_prices=("discount_price", "nunique"),
        different_actual_prices=("actual_price", "nunique"),
        different_ratings=("review_rating", "nunique")
    )
    .query("occurrences > 1")
    .sort_values("occurrences", ascending=False)
)

print("Repeated product names:", len(repeated_analysis))

print(
    "Names with different discount prices:",
    (repeated_analysis["different_discount_prices"] > 1).sum()
)

print(
    "Names with different actual prices:",
    (repeated_analysis["different_actual_prices"] > 1).sum()
)

print(
    "Names with different ratings:",
    (repeated_analysis["different_ratings"] > 1).sum()
)

display(repeated_analysis.head(20))

Repeated product names: 288
Names with different discount prices: 67
Names with different actual prices: 40
Names with different ratings: 137
                                                                                                                               occurrences  different_discount_prices  different_actual_prices  different_ratings
name                                                                                                                                                                                                             
TAGG Verve Sense Smartwatch with 1.70'' Large Display, Real SPO2, and Real-Time Heart Rate Tracking, 7 Days Battery Backu...             7                          1                        1                  3
Fire-Boltt Visionary 1.78" AMOLED Bluetooth Calling Smartwatch with 368 * 448 Pixel Resolution, Rotating Crown & 60Hz Refr...            6                          1                        2                  1
Fire-Boltt Eterno 